In [0]:
%pip install torch

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install h3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.9/974.9 kB 24.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, col, pandas_udf
from pyspark.sql.types import DoubleType, StringType, IntegerType
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

spark = SparkSession.builder \
    .appName("TerrainRiskScoring") \
    .getOrCreate()


# ── 1. READ SILVER TABLE ─────────────────────────────────────────────────────

def load_silver(table_name: str = "sima_rakshak_catalog.silver.hex_unified_live"):
    df = spark.table(table_name)

    print(f"Loaded {df.count()} hexagons")
    print("Schema:")
    df.printSchema()

    expected = {
        'hex_id', 'elevation_m', 'snow_depth_cm', 'precip_mm',
        'visibility_m', 'cloud_pct', 'temp_c',
        'wind_ms', 'inclination_deg', 'denseness', 'roughness_idx'
    }
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in silver table: {missing}")

    return df


# ── 2. FEATURE ENGINEERING IN SPARK ─────────────────────────────────────────

def build_features(df):
    """
    All transformations done in Spark — stays distributed,
    nothing collected to driver yet.

    Directionality rules:
      HIGH value = HIGH surveillance difficulty = HIGH infiltration risk

      Direct:   elevation_m, snow_depth_cm, precip_mm, cloud_pct,
                wind_ms, inclination_deg, denseness, roughness_idx
      Inverted: visibility_m  (high visibility = easy to see = LOW risk)
                temp_c        (high temp = open terrain = LOWER risk)
    """

    VIS_MAX  = 10000.0
    TEMP_MIN = -20.0
    TEMP_MAX =  50.0

    df = df.select(
        col('hex_id'),

        # --- Direct difficulty features ---
        col('elevation_m')                                      .alias('elevation'),
        col('snow_depth_cm')                                    .alias('snow'),
        col('precip_mm')                                        .alias('precipitation'),
        col('cloud_pct')                                        .alias('cloud'),
        col('wind_ms')                                          .alias('windspeed'),
        col('inclination_deg')                                  .alias('inclination'),
        col('denseness')                                        .alias('denseness'),
        col('roughness_idx')                                    .alias('roughness'),

        # --- Inverted features ---
        (F.lit(1.0) - (
            F.least(col('visibility_m'), F.lit(VIS_MAX)) / F.lit(VIS_MAX)
        ))                                                      .alias('low_visibility'),

        (F.lit(1.0) - (
            (F.least(F.greatest(col('temp_c'), F.lit(TEMP_MIN)), F.lit(TEMP_MAX))
             - F.lit(TEMP_MIN)) / F.lit(TEMP_MAX - TEMP_MIN)
        ))                                                      .alias('cold_temp_risk'),

        # --- Interaction features ---
        (col('denseness') *
         (F.lit(1.0) - F.least(col('visibility_m'), F.lit(VIS_MAX)) / F.lit(VIS_MAX))
        )                                                       .alias('conceal_x_lowvis'),

        (col('snow_depth_cm') * col('inclination_deg'))         .alias('snow_x_slope'),

        (col('cloud_pct') * col('precip_mm') / F.lit(100.0))   .alias('cloud_x_rain'),

        (col('roughness_idx') * col('inclination_deg'))         .alias('rough_x_slope'),
    )

    return df


FEATURE_COLS = [
    'elevation', 'snow', 'precipitation', 'cloud', 'windspeed',
    'inclination', 'denseness', 'roughness', 'low_visibility',
    'cold_temp_risk', 'conceal_x_lowvis', 'snow_x_slope',
    'cloud_x_rain', 'rough_x_slope'
]


# ── 3. COLLECT TO DRIVER FOR MODEL TRAINING ──────────────────────────────────

def collect_for_training(spark_df):
    print("Collecting to driver for model training...")
    pdf = spark_df.select(['hex_id'] + FEATURE_COLS).toPandas()

    for col_name in FEATURE_COLS:
        median_val = pdf[col_name].median()
        null_count = pdf[col_name].isna().sum()
        if null_count > 0:
            print(f"  Filling {null_count} nulls in '{col_name}' with median {median_val:.3f}")
            pdf[col_name] = pdf[col_name].fillna(median_val)

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(pdf[FEATURE_COLS])

    print(f"Ready: {X_scaled.shape[0]} hexagons, {X_scaled.shape[1]} features")
    return pdf, X_scaled, scaler


# ── 4. DIRECTED AUTOENCODER ──────────────────────────────────────────────────

class TerrainAutoencoder(nn.Module):
    def __init__(self, input_dim=14, bottleneck_dim=3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 7),
            nn.ReLU(),
            nn.Linear(7, bottleneck_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 7),
            nn.ReLU(),
            nn.Linear(7, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z


def difficulty_proxy(X: torch.Tensor) -> torch.Tensor:
    """
    Column order matches FEATURE_COLS:
    0:elevation       1:snow            2:precipitation
    3:cloud           4:windspeed       5:inclination
    6:denseness       7:roughness       8:low_visibility
    9:cold_temp_risk  10:conceal_x_lowvis 11:snow_x_slope
    12:cloud_x_rain   13:rough_x_slope
    """
    weights = torch.tensor([
        0.07,   # elevation
        0.07,   # snow
        0.05,   # precipitation
        0.07,   # cloud
        0.04,   # windspeed
        0.10,   # inclination
        0.10,   # denseness
        0.10,   # roughness
        0.10,   # low_visibility
        0.07,   # cold_temp_risk
        0.09,   # conceal_x_lowvis
        0.05,   # snow_x_slope
        0.04,   # cloud_x_rain
        0.05,   # rough_x_slope
    ], dtype=torch.float32)

    return (X * weights).sum(dim=1)


def combined_loss(recon, original, z, alpha=0.4):
    recon_loss  = nn.MSELoss()(recon, original)

    difficulty  = difficulty_proxy(original)
    z_magnitude = torch.norm(z, dim=1)

    diff_norm = (difficulty  - difficulty.mean())  / (difficulty.std()  + 1e-8)
    zmag_norm = (z_magnitude - z_magnitude.mean()) / (z_magnitude.std() + 1e-8)

    direction_loss = nn.MSELoss()(zmag_norm, diff_norm)
    return recon_loss + alpha * direction_loss


def train_autoencoder(X_scaled: np.ndarray,
                      epochs: int = 400,
                      lr: float = 1e-3,
                      alpha: float = 0.4) -> TerrainAutoencoder:

    X_tensor  = torch.FloatTensor(X_scaled)
    model     = TerrainAutoencoder(input_dim=X_scaled.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=100, gamma=0.5
    )

    print(f"\nTraining autoencoder on CPU...")
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        recon, z = model(X_tensor)
        loss = combined_loss(recon, X_tensor, z, alpha)
        loss.backward()
        optimizer.step()
        scheduler.step()

        if epoch % 100 == 0:
            print(f"  Epoch {epoch:4d} | Loss: {loss.item():.5f}")

    print("Training complete.")
    return model


def compute_risk_scores(model: TerrainAutoencoder,
                        X_scaled: np.ndarray) -> np.ndarray:
    model.eval()
    X_tensor = torch.FloatTensor(X_scaled)
    with torch.no_grad():
        _, z      = model(X_tensor)
        magnitude = torch.norm(z, dim=1).numpy()

    scores = (magnitude - magnitude.min()) / \
             (magnitude.max() - magnitude.min()) * 100
    return scores


# ── 5. WRITE RESULTS BACK TO SPARK / GOLD TABLE ──────────────────────────────

def write_to_gold(pdf: pd.DataFrame,
                  table_name: str = "sima_rakshak_catalog.gold.hex_risk_scores"):
    # Parse catalog and schema from table name
    parts = table_name.split('.')
    if len(parts) == 3:
        catalog_name, schema_name, _ = parts
        # Create schema if it doesn't exist
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
    
    output_cols = ['hex_id', 'risk_score'] + FEATURE_COLS

    result_spark = spark.createDataFrame(pdf[output_cols])

    result_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

    print(f"\nResults written to: {table_name}")
    print(f"Total hexes: {result_spark.count()}")


# ── 6. FULL PIPELINE ─────────────────────────────────────────────────────────

def run_pipeline(
    silver_table : str   = "sima_rakshak_catalog.silver.hex_unified_live",
    gold_table   : str   = "sima_rakshak_catalog.gold.hex_risk_scores",
    alpha        : float = 0.4,
    epochs       : int   = 400,
):
    # 1. Load from silver
    spark_df = load_silver(silver_table)

    # 2. Feature engineering in Spark
    spark_df = build_features(spark_df)

    # 3. Collect to driver
    pdf, X_scaled, scaler = collect_for_training(spark_df)

    # 4. Train autoencoder
    model = train_autoencoder(X_scaled, epochs=epochs, alpha=alpha)

    # 5. Score hexagons
    pdf['risk_score'] = compute_risk_scores(model, X_scaled)

    # 6. Write gold table
    write_to_gold(pdf, gold_table)

    # 7. Full summary — all 75 hexes ranked
    print("\nAll hexagons ranked by risk score:")
    print(
        pdf[['hex_id', 'risk_score']]
        .sort_values('risk_score', ascending=False)
        .to_string(index=False)
    )

    return pdf, model, scaler


# ── ENTRY POINT ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    pdf, model, scaler = run_pipeline(
        silver_table = "sima_rakshak_catalog.silver.hex_unified_live",
        gold_table   = "sima_rakshak_catalog.gold.hex_risk_scores",
        alpha        = 0.4,
        epochs       = 400,
    )

Loaded 75 hexagons
Schema:
root
 |-- hex_id: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- elevation_m: double (nullable = true)
 |-- inclination_deg: double (nullable = true)
 |-- roughness_idx: double (nullable = true)
 |-- denseness: double (nullable = true)
 |-- temp_c: double (nullable = true)
 |-- wind_ms: double (nullable = true)
 |-- precip_mm: double (nullable = true)
 |-- visibility_m: double (nullable = true)
 |-- cloud_pct: double (nullable = true)
 |-- snow_depth_cm: double (nullable = true)
 |-- observation_time: timestamp (nullable = true)
 |-- processed_at: timestamp (nullable = true)

Ready: 75 hexagons, 14 features

Training autoencoder on CPU...
  Epoch    0 | Loss: 0.59382
  Epoch  100 | Loss: 0.14297
  Epoch  200 | Loss: 0.12262
  Epoch  300 | Loss: 0.11481
Training complete.

Results written to: sima_rakshak_catalog.gold.hex_risk_scores
Total hexes: 75

All hexagons ranked by risk score:
  hex_id  risk_score
HE

In [0]:
df = spark.table("sima_rakshak_catalog.silver.hex_unified_live")
display(df)

hex_id,lat,lon,elevation_m,inclination_deg,roughness_idx,denseness,temp_c,wind_ms,precip_mm,visibility_m,cloud_pct,snow_depth_cm,observation_time,processed_at
HEX_0000,32.52862230401004,74.87194850928567,281.0,1.0,0.23190312952151804,0.8332440696273506,29.4,3.944444444444444,0.0,10285.5,17.025,0.0,2026-04-18T05:04:32.037Z,2026-04-18T05:05:08.023Z
HEX_0001,32.51563204806692,74.86452236041224,282.0,2.0,0.5806549588841793,0.7801288880544629,29.4,3.944444444444444,0.0,9333.0,26.05,0.0,2026-04-18T05:04:32.480Z,2026-04-18T05:05:08.023Z
HEX_0002,32.502641332714035,74.85709834226839,277.0,17.0,2.397256934967401,0.7235716874769464,29.5,3.944444444444444,0.0,9344.5,31.925,0.0,2026-04-18T05:04:32.926Z,2026-04-18T05:05:08.023Z
HEX_0003,32.503026017644146,74.87013549001699,282.0,2.0,0.564533520605624,0.8639936580294625,29.4,3.944444444444444,0.0,8977.0,22.05,0.0,2026-04-18T05:04:33.368Z,2026-04-18T05:05:08.023Z
HEX_0004,32.507151525255104,74.85381285939958,276.0,16.0,2.2064811302790672,0.8350550096619855,29.5,3.944444444444444,0.0,9199.0,29.9,0.0,2026-04-18T05:04:33.810Z,2026-04-18T05:05:08.023Z
HEX_0005,32.51127490813735,74.83748874606528,274.0,14.0,1.9168580615084903,0.7121190269202857,29.5,3.944444444444444,0.0,10144.0,30.85,0.0,2026-04-18T05:04:34.259Z,2026-04-18T05:05:08.023Z
HEX_0006,32.500487520400185,74.80018354330345,269.0,9.0,1.3516622049901792,0.7039551045172018,28.9,4.111111111111112,0.0,8819.5,25.725,0.0,2026-04-18T05:04:34.701Z,2026-04-18T05:05:08.023Z
HEX_0007,32.4868805272618,74.79450240384018,270.0,10.0,1.5267589063532805,0.7980617352107464,28.9,4.111111111111112,0.0,9586.0,31.75,0.0,2026-04-18T05:04:35.147Z,2026-04-18T05:05:08.023Z
HEX_0008,32.495564417844164,74.79629547803465,270.0,10.0,1.5067505985131322,0.8722533183390262,28.9,4.111111111111112,0.0,9799.0,17.75,0.0,2026-04-18T05:04:35.590Z,2026-04-18T05:05:08.023Z
HEX_0009,32.49493432198676,74.77928039481506,267.0,7.0,1.2176958696835711,0.8366146154483923,29.0,4.111111111111112,0.0,9652.5,28.675,0.0,2026-04-18T05:04:36.037Z,2026-04-18T05:05:08.023Z


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, col, pandas_udf
from pyspark.sql.types import DoubleType, StringType, IntegerType
# Step 1 — create the schema
spark.sql("CREATE SCHEMA IF NOT EXISTS sima_rakshak_catalog.gold")
print("Gold schema created.")

# Step 2 — create the table
spark.sql("""
    CREATE TABLE IF NOT EXISTS sima_rakshak_catalog.gold.hex_risk_scores (
        hex_id           STRING,
        risk_score       DOUBLE,
        elevation        DOUBLE,
        snow             DOUBLE,
        precipitation    DOUBLE,
        cloud            DOUBLE,
        windspeed        DOUBLE,
        inclination      DOUBLE,
        denseness        DOUBLE,
        roughness        DOUBLE,
        low_visibility   DOUBLE,
        cold_temp_risk   DOUBLE,
        conceal_x_lowvis DOUBLE,
        snow_x_slope     DOUBLE,
        cloud_x_rain     DOUBLE,
        rough_x_slope    DOUBLE
    )
    USING DELTA
""")
print("Gold table created.")

Gold schema created.
Gold table created.


In [0]:
# Add lat and long columns to gold table from silver table

# Step 1: Add columns only if they don't exist
try:
    spark.sql("""
        ALTER TABLE sima_rakshak_catalog.gold.hex_risk_scores
        ADD COLUMNS (
            latitude DOUBLE,
            longitude DOUBLE
        )
    """)
    print("Added latitude and longitude columns to gold table.")
except Exception as e:
    if "FIELD_ALREADY_EXISTS" in str(e):
        print("Latitude and longitude columns already exist, skipping ADD COLUMNS.")
    else:
        raise

# Step 2: Update gold table with lat/long from silver table
spark.sql("""
    MERGE INTO sima_rakshak_catalog.gold.hex_risk_scores AS gold
    USING sima_rakshak_catalog.silver.hex_unified_live AS silver
    ON gold.hex_id = silver.hex_id
    WHEN MATCHED THEN UPDATE SET
        gold.latitude  = silver.lat,
        gold.longitude = silver.lon
""")
print("Updated latitude and longitude values in gold table.")

Latitude and longitude columns already exist, skipping ADD COLUMNS.
Updated latitude and longitude values in gold table.


In [0]:
gold_df = spark.table("sima_rakshak_catalog.gold.hex_risk_scores")
display(gold_df)

hex_id,risk_score,elevation,snow,precipitation,cloud,windspeed,inclination,denseness,roughness,low_visibility,cold_temp_risk,conceal_x_lowvis,snow_x_slope,cloud_x_rain,rough_x_slope,latitude,longitude
HEX_0000,8.086757,281.0,0.0,0.0,17.025,3.944444444444444,1.0,0.8332440696273506,0.23190312952151804,0.0,0.29428571428571426,0.0,0.0,0.0,0.23190312952151804,32.52862230401004,74.87194850928567
HEX_0001,34.80266,282.0,0.0,0.0,26.05,3.944444444444444,2.0,0.7801288880544629,0.5806549588841793,0.06669999999999998,0.29428571428571426,0.05203459683323266,0.0,0.0,1.1613099177683586,32.51563204806692,74.86452236041224
HEX_0002,74.82309,277.0,0.0,0.0,31.925,3.944444444444444,17.0,0.7235716874769464,2.397256934967401,0.06555,0.2928571428571428,0.04743012411411384,0.0,0.0,40.753367894445816,32.502641332714035,74.85709834226839
HEX_0003,54.86397,282.0,0.0,0.0,22.05,3.944444444444444,2.0,0.8639936580294625,0.564533520605624,0.10229999999999995,0.29428571428571426,0.08838655121641396,0.0,0.0,1.129067041211248,32.503026017644146,74.87013549001699
HEX_0004,86.03325,276.0,0.0,0.0,29.9,3.944444444444444,16.0,0.8350550096619855,2.2064811302790672,0.08009999999999995,0.2928571428571428,0.066887906273925,0.0,0.0,35.303698084465076,32.507151525255104,74.85381285939958
HEX_0005,44.054596,274.0,0.0,0.0,30.85,3.944444444444444,14.0,0.7121190269202857,1.9168580615084903,0.0,0.2928571428571428,0.0,0.0,0.0,26.836012861118864,32.51127490813735,74.83748874606528
HEX_0006,57.76792,269.0,0.0,0.0,25.725,4.111111111111112,9.0,0.7039551045172018,1.3516622049901792,0.11804999999999999,0.3014285714285715,0.08310190008825566,0.0,0.0,12.164959844911612,32.500487520400185,74.80018354330345
HEX_0007,58.8828,270.0,0.0,0.0,31.75,4.111111111111112,10.0,0.7980617352107464,1.5267589063532805,0.04139999999999999,0.3014285714285715,0.03303975583772489,0.0,0.0,15.267589063532805,32.4868805272618,74.79450240384018
HEX_0008,46.253933,270.0,0.0,0.0,17.75,4.111111111111112,10.0,0.8722533183390262,1.5067505985131322,0.020100000000000007,0.3014285714285715,0.017532291698614432,0.0,0.0,15.067505985131323,32.495564417844164,74.79629547803465
HEX_0009,49.532345,267.0,0.0,0.0,28.675,4.111111111111112,7.0,0.8366146154483923,1.2176958696835711,0.03474999999999995,0.30000000000000004,0.02907235788683159,0.0,0.0,8.523871087784999,32.49493432198676,74.77928039481506
